# GraphRAG Multi-Hop Retrieval

Design a mock GraphRAG ingestion schema using a graph database layer to retrieve multi-hop relational context for knowledge-augmented generation.

In [1]:
import networkx as nx
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

## Mock Data and Extracted Triples Generation

In [2]:
# Raw text representing the unstructured knowledge
with open("corporate_history.txt", "w") as f:
    f.write("Alice heads the Engineering Department. ")
    f.write("The Engineering Department is responsible for Project Titan. ")
    f.write("Project Titan relies on the new Quantum Server. ")
    f.write("The Quantum Server is located in the Berlin Data Center.")

# Mock extracted knowledge triples (Subject, Predicate, Object) representing an NLP extraction pipeline
triples_data = [
    ("Alice", "HEADS", "Engineering Department"),
    ("Engineering Department", "RESPONSIBLE_FOR", "Project Titan"),
    ("Project Titan", "RELIES_ON", "Quantum Server"),
    ("Quantum Server", "LOCATED_IN", "Berlin Data Center"),
]

df_triples = pd.DataFrame(triples_data, columns=["Subject", "Predicate", "Object"])
df_triples.to_csv("extracted_triples.csv", index=False)

## Graph Database Ingestion Layer

In [3]:
# Using NetworkX as an in-memory Graph Database layer
def ingest_into_graph(csv_path):
    df = pd.read_csv(csv_path)
    G = nx.DiGraph()

    for _, row in df.iterrows():
        G.add_node(row["Subject"])
        G.add_node(row["Object"])
        G.add_edge(row["Subject"], row["Object"], relation=row["Predicate"])

    return G


knowledge_graph = ingest_into_graph("extracted_triples.csv")
print(
    f"Graph ingested with {knowledge_graph.number_of_nodes()} nodes and {knowledge_graph.number_of_edges()} edges."
)

Graph ingested with 5 nodes and 4 edges.


## Multi-Hop Relational Context Retrieval

In [4]:
def retrieve_multi_hop_context(graph, start_entity, max_depth=2):
    if start_entity not in graph:
        return "Entity not found in the Knowledge Graph."

    context_triples = []

    # Perform a Breadth-First Search (BFS) to gather N-hop neighbors
    edges = nx.bfs_edges(graph, source=start_entity, depth_limit=max_depth)

    for u, v in edges:
        relation = graph[u][v]["relation"]
        context_triples.append(f"{u} -[{relation}]-> {v}")

    return "\n".join(context_triples)


query_entity = "Alice"
graph_context = retrieve_multi_hop_context(knowledge_graph, query_entity, max_depth=3)

print("=== Retrieved Multi-Hop Graph Context ===")
print(graph_context)

=== Retrieved Multi-Hop Graph Context ===
Alice -[HEADS]-> Engineering Department
Engineering Department -[RESPONSIBLE_FOR]-> Project Titan
Project Titan -[RELIES_ON]-> Quantum Server


## Knowledge-Augmented Generation (GraphRAG)

In [5]:
llm = ChatOllama(model="lfm3:1.7b", temperature=0)

template = """You are an AI assistant powered by a Knowledge Graph.
Use the following relational graph context to answer the user's question.

Graph Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate.from_template(template)

graph_rag_chain = prompt | llm | StrOutputParser()

question = "How is Alice connected to the Berlin Data Center?"

print("\n=== Final GraphRAG Output ===")
response = graph_rag_chain.invoke({"context": graph_context, "question": question})
print(response)


=== Final GraphRAG Output ===
Based on the supplied relational graph, the links are:

- **Alice** → *heads* → **Engineering Department**
- **Engineering Department** → *responsible for* → **Project Titan**
- **Project Titan** → *relies on* → **Quantum Server**

There is no entry that connects any of these nodes (Alice, Engineering Department, Project Titan, or Quantum Server) to a “Berlin Data Center.” Consequently, within the given graph there is no recorded path or relationship linking Alice to the Berlin Data Center. 

**Therefore, according to the provided information, Alice’s connection to the Berlin Data Center cannot be determined.**
